# Day 19 Tutorial：ESOL 上的第一条 MLP Pipeline

> **课程附带教程，不是学习者实验记录。** 数据是公开 ESOL，目标是水溶解度 logS，不是粘合剂性能。

## Goal

在固定 scaffold train/valid 和 1024 维 ECFP 上，只建立并定位第一条插补—缩放—MLP Pipeline，读取训练/验证指标与迭代诊断。Dummy 和传统模型的正式同协议比较留到 Day 21。


## Setup

从仓库缓存加载 ESOL；首次运行可能由 DeepChem 下载公开数据。标签保持原始 logS，测试集不参与。本日只学习 MLP 的输入、输出和诊断，不据此宣布算法优劣。


In [1]:
from pathlib import Path
import contextlib
import io

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-practice 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)
valid_ids = np.asarray(valid_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)
assert X_valid.shape == (113, 1024) and y_valid.shape == (113,)
print("Task:", tasks[0])
print("Train / valid:", X_train.shape, X_valid.shape)
print("测试集对象保持封存，本教程不创建测试预测。")


Task: measured log solubility in mols per litre
Train / valid: (902, 1024) (113, 1024)
测试集对象保持封存，本教程不创建测试预测。


In [2]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def regression_metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(root_mean_squared_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


## Steps

### 1. 创建并拟合第一条 MLP

`hidden_layer_sizes=(32,)` 是一个 32 单元隐藏层。`max_iter=120` 是显式的快速教学上限；Day 20 再研究正则化和早停。


In [3]:
mlp_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=120,
        early_stopping=False,
        random_state=SEED,
    ),
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    mlp_pipeline.fit(X_train, y_train)

convergence_messages = [
    str(item.message)
    for item in caught
    if issubclass(item.category, ConvergenceWarning)
]
print("Convergence warnings:", convergence_messages or "none")


Convergence warnings: none


### 2. 读取 MLP 自身的训练与验证诊断

训练指标用于判断拟合程度，外部验证指标描述当前一次固定协议。今天不把这些结果与其他算法排成排行榜。


In [4]:
rows = []
for split, X_part, y_part in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
]:
    prediction = mlp_pipeline.predict(X_part)
    rows.append({"split": split, **regression_metrics(y_part, prediction)})

metrics_table = pd.DataFrame(rows)
trained_mlp = mlp_pipeline.named_steps["mlpregressor"]
display(metrics_table.round(4))
print("n_iter_:", trained_mlp.n_iter_)
print("last five training losses:", np.round(trained_mlp.loss_curve_[-5:], 6))


,split,mae,rmse,r2
0,train,0.0974,0.3107,0.9774
1,valid,2.5048,3.0679,-1.4085


n_iter_: 116
last five training losses: [0.047766 0.053062 0.054389 0.048655 0.052389]


## Checks

下列断言验证形状、指标、Pipeline 顺序和拟合后属性。它们不证明 MLP 胜过其他算法，也不证明模型适合粘合剂任务。


In [5]:
assert y_train.ndim == 1 and y_valid.ndim == 1
assert list(mlp_pipeline.named_steps) == [
    "simpleimputer", "standardscaler", "mlpregressor"
]
assert len(trained_mlp.loss_curve_) == trained_mlp.n_iter_
assert np.isfinite(metrics_table[["mae", "rmse", "r2"]]).all().all()
assert set(metrics_table["split"]) == {"train", "valid"}
print("Checks passed: 仅定位 MLP；测试集仍封存。")


Checks passed: 仅定位 MLP；测试集仍封存。


## Next Steps

Day 20 比较正则化与内部早停；Day 21 才把冻结后的 MLP 与 Dummy 和 Day 07 传统模型放入同一协议。不要把本教程的保存输出当作你已经完成的实验。
